# Classification Trees und Ensembles — Konzepte

Lies die Idee, die Formel und das kleine Beispiel in dieser Reihenfolge. Der Demonstrationscode ist vorhanden.

## Libraries und Einstellungen

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import confusion_matrix, roc_auc_score

## 1. Einen classification tree lesen

Ein interner Knoten stellt eine Ja/Nein-Frage. Die **root** ist der erste Knoten, und ein **leaf** ist ein Endknoten. Die Tiefe ist der längste Pfad von der root zum leaf, gemessen in Kanten.

Ein leaf schätzt die Klassenwahrscheinlichkeiten anhand der Klassenanteile im Training:
$$\hat P(y=k\mid\mathrm{leaf})=\frac{\mathrm{number\ in\ class}\ k}{\mathrm{number\ in\ leaf}}.$$
Die vorhergesagte Klasse ist die Klasse mit dem höchsten Anteil. Bei 145 Überlebenden und 125 Nichtüberlebenden gilt $P(\mathrm{survived})=145/270\approx0.537$; daher lautet die vorhergesagte Klasse „survived“. Ein tree sagt sowohl Wahrscheinlichkeiten als auch Labels voraus.

## 2. Gini: Wie gemischt ist ein Knoten?

Für Klassenanteile $p_k$ gilt:
$$G=1-\sum_k p_k^2.$$
Für zwei Klassen gilt $G=1-p^2-(1-p)^2$.

Ein reiner Knoten hat Gini 0; 
- 100 % survived
- 0 % not survived

eine gleichmässige 50/50-Mischung hat Gini 0.5. 

Entropy ist eine Alternative:
$$H=-\sum_k p_k\log(p_k),$$
wobei ein Anteil von null den Wert null beiträgt. Beide bevorzugen reinere child nodes.

Ein split wird anhand der grössengewichteten impurity der child nodes beurteilt:
$$\Delta G=G_{\mathrm{parent}}-\left(\frac{n_L}{n}G_L+\frac{n_R}{n}G_R\right).$$
$n_L,n_R$ sind die Grössen der child nodes; $n=n_L+n_R$. Wähle die grösste Abnahme und wiederhole den Vorgang rekursiv. Beispiel: Der parent hat 40 Überlebende und 60 Nichtüberlebende ($G=0.48$). 

Der **split „male?“** ergibt einen gewichteten Gini von **0.36** und eine **Abnahme von 0.12**; der alternative split nach first-class ergibt eine kleinere Abnahme.

In [5]:
p = 0.4
parent_gini = 1 - p ** 2 - (1 - p) ** 2
print('Parent Gini:', parent_gini)
g_left = 1 - 0.2 ** 2 - 0.8 ** 2
g_right = 1 - 0.7 ** 2 - 0.3 ** 2
weighted = 0.6 * g_left + 0.4 * g_right
print('Weighted child Gini:', round(weighted, 3))
print('Decrease:', round(parent_gini - weighted, 3))

Parent Gini: 0.48
Weighted child Gini: 0.36
Decrease: 0.12


Der Baum sucht nach einem split, der den Gini-Wert möglichst stark reduziert.

- Vor dem split:        Gini = 0.48
- Nach dem split:       Gini = 0.36
- Verbesserung:         0.12

## 3. Wachsen, stoppen und prune

Probiere mögliche Eingaben und Schwellenwerte aus, behalte die grösste impurity-Abnahme und wiederhole den Vorgang in den child nodes. Stoppe, wenn ein Knoten rein ist oder eine Einstellung wie `max_depth` oder `min_samples_leaf` das Aufteilen verhindert.

Pruning fügt pro leaf eine Strafe hinzu, genau wie bei regression trees:
$$\mathrm{objective}=\mathrm{tree\ impurity}+\alpha\times\mathrm{number\ of\ leaves}.$$
Ein grösseres `ccp_alpha` kann Äste entfernen. Tiefe trees können overfitten; wähle tree-Einstellungen mit CV und nicht allein anhand der training accuracy.

## 4. Random forest und boosting

Ein random forest erstellt aus bootstrap samples und zufälligen feature subsets bei jedem split viele trees. Scikit-learn bildet den Durchschnitt der Klassenwahrscheinlichkeiten:
$$\hat p=\frac1B\sum_{b=1}^B\hat p_b.$$
Für 0.8, 0.6 und 0.3 beträgt der Mittelwert ungefähr 0.567: Bei einem threshold von 0.5 wird survived vorhergesagt. Klassisches bagging wird oft als voting beschrieben; das Mitteln von Klassenwahrscheinlichkeiten kann zu einem anderen Ergebnis führen als das voting anhand der tree labels.

Gradient boosting erstellt sequenziell kleine trees, um die aktuellen Vorhersagen zu verbessern. Bei classification arbeitet es mit log-loss und pseudo-residuals und nicht einfach mit den 0/1-Fehlern. Die Prinzipien sind dieselben wie bei regression, aber die Ausgaben beschreiben Klassenwahrscheinlichkeiten.

In [3]:
tree_probabilities = np.array([0.8, 0.6, 0.3])
print('Forest probability:', tree_probabilities.mean())
print('Predicted class:', int(tree_probabilities.mean() >= 0.5))

Forest probability: 0.5666666666666667
Predicted class: 1


## 5. Alles zusammenführen: ein kleines GridSearchCV

Das abschliessende Beispiel kombiniert train/test splitting, CV, einen forest und AUC. `GridSearchCV` probiert eine kleine Liste von Einstellungen aus und behält jene mit dem höchsten durchschnittlichen validation score.

In der Übung ist die vollständige Suche vorgegeben: zwei `max_features`-Werte und drei `min_samples_leaf`-Werte, ausgewertet mit 5-fold CV auf den Trainingszeilen. Du erklärst das Ergebnis, statt die Suche zu programmieren. Erst nach der Suche werten wir die Testzeilen aus. AUC verwendet Wahrscheinlichkeiten; eine confusion matrix verwendet Labels bei einem threshold.

### Jupyter notebook — Footer-Informationen

Füge diese Informationen ein, wenn du dein Notebook einreichst.

In [4]:
import platform
from datetime import datetime
print(platform.platform())
print("Python:", platform.python_version())
print("Date:", datetime.now().isoformat(timespec="seconds"))

Linux-7.0.14-linuxkit-aarch64-with-glibc2.41
Python: 3.11.17
Date: 2026-10-27T18:42:18
